Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

Load fresh dataset

In [2]:
df = pd.read_csv("medical_eligibility_dataset.csv")

df.head()

,Donor_ID,Full_Name,Gender,Age,Blood_Group,Contact_Number,Email,City,State,Country,Last_Donation_Date,Total_Donations,Eligible_for_Donation,Medical_Condition,Weight_kg,Hemoglobin_g_dL,Donation_Center,Registration_Date
0,DNR000001,Sangeeta Menon,Female,38,O+,1819600042,sangeeta.menon8280@gmail.com,Kolkata,West Bengal,India,07-10-2025,1,Yes,NaN,66.6,13.6,Red Cross Blood Bank,02-07-2021
1,DNR000002,Meena Iyer,Female,49,B+,265423420,meena.iyer6225@gmail.com,Jaipur,Rajasthan,India,08-11-2020,1,No,Hypertension,70.8,14.0,Metro Blood Bank,03-03-2023
2,DNR000003,Priya Nair,Female,29,B+,1849593012,priya.nair4742@gmail.com,Gurgaon,Haryana,India,12-04-2025,2,No,Diabetes,73.4,12.5,Fortis Blood Bank,15-10-2015
3,DNR000004,Vijay Kapoor,Male,29,O+,3419283185,vijay.kapoor4423@gmail.com,Thiruvananthapuram,Kerala,India,21-02-2025,1,Yes,NaN,57.9,14.8,NABL Blood Centre,09-05-2022
4,DNR000005,Rahul Iyer,Male,27,A+,6413953676,rahul.iyer2341@gmail.com,Bhopal,Madhya Pradesh,India,18-04-2024,1,Yes,NaN,74.0,17.1,NABL Blood Centre,13-07-2022


Basic inspection

In [3]:
print(df.shape)
df.info()

(10000, 18)
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Donor_ID               10000 non-null  str    
 1   Full_Name              10000 non-null  str    
 2   Gender                 10000 non-null  str    
 3   Age                    10000 non-null  int64  
 4   Blood_Group            10000 non-null  str    
 5   Contact_Number         10000 non-null  int64  
 6   Email                  10000 non-null  str    
 7   City                   10000 non-null  str    
 8   State                  10000 non-null  str    
 9   Country                10000 non-null  str    
 10  Last_Donation_Date     10000 non-null  str    
 11  Total_Donations        10000 non-null  int64  
 12  Eligible_for_Donation  10000 non-null  str    
 13  Medical_Condition      2195 non-null   str    
 14  Weight_kg              10000 non-null  float64
 15  He

Clean target

In [4]:
df["Eligible_for_Donation"] = df["Eligible_for_Donation"].map({
    "Yes": 1,
    "No": 0
})

Clean medical condition

In [5]:
df["Medical_Condition"] = df["Medical_Condition"].fillna("None")

Do NOT use dates for now

In [6]:
drop_cols = [
    "Donor_ID",
    "Full_Name",
    "Contact_Number",
    "Email",
    "City",
    "State",
    "Country",
    "Donation_Center",
    "Last_Donation_Date",
    "Registration_Date"
]

drop_cols = [c for c in drop_cols if c in df.columns]

df_clean = df.drop(columns=drop_cols)

df_clean.head()

,Gender,Age,Blood_Group,Total_Donations,Eligible_for_Donation,Medical_Condition,Weight_kg,Hemoglobin_g_dL
0,Female,38,O+,1,1,None,66.6,13.6
1,Female,49,B+,1,0,Hypertension,70.8,14.0
2,Female,29,B+,2,0,Diabetes,73.4,12.5
3,Male,29,O+,1,1,None,57.9,14.8
4,Male,27,A+,1,1,None,74.0,17.1


Confirm clean columns

In [7]:
df_clean.info()
df_clean.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Gender                 10000 non-null  str    
 1   Age                    10000 non-null  int64  
 2   Blood_Group            10000 non-null  str    
 3   Total_Donations        10000 non-null  int64  
 4   Eligible_for_Donation  10000 non-null  int64  
 5   Medical_Condition      10000 non-null  str    
 6   Weight_kg              10000 non-null  float64
 7   Hemoglobin_g_dL        10000 non-null  float64
dtypes: float64(2), int64(3), str(3)
memory usage: 745.6 KB


Gender                   0
Age                      0
Blood_Group              0
Total_Donations          0
Eligible_for_Donation    0
Medical_Condition        0
Weight_kg                0
Hemoglobin_g_dL          0
dtype: int64

Prepare model data

In [8]:
X = df_clean.drop(columns=["Eligible_for_Donation"])
y = df_clean["Eligible_for_Donation"]

X = pd.get_dummies(X, drop_first=True)
X = X.fillna(0)

print(X.shape)
X.head()

(10000, 20)


,Age,Total_Donations,Weight_kg,Hemoglobin_g_dL,Gender_Male,Gender_Other,Blood_Group_A-,Blood_Group_AB+,Blood_Group_AB-,Blood_Group_B+,Blood_Group_B-,Blood_Group_O+,Blood_Group_O-,Medical_Condition_Asthma,Medical_Condition_Cardiac issues,Medical_Condition_Diabetes,Medical_Condition_Hypertension,Medical_Condition_Infection (resolved),Medical_Condition_None,Medical_Condition_Recent Surgery
0,38,1,66.6,13.6,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False
1,49,1,70.8,14.0,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False
2,29,2,73.4,12.5,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False
3,29,1,57.9,14.8,True,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False
4,27,1,74.0,17.1,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False


Train clean model

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, predictions))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predictions))
print("\nClassification Report:")
print(classification_report(y_test, predictions))

Accuracy: 1.0

Confusion Matrix:
[[ 717    0]
 [   0 1283]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       717
           1       1.00      1.00      1.00      1283

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



Test if dataset is rule-generated

In [10]:
rule_prediction = (
    (df["Medical_Condition"] == "None") &
    (df["Weight_kg"] >= 50) &
    (df["Hemoglobin_g_dL"] >= 12.5)
).astype(int)

print("Rule Accuracy:", (rule_prediction == df["Eligible_for_Donation"]).mean())

pd.crosstab(rule_prediction, df["Eligible_for_Donation"])

Rule Accuracy: 1.0


Eligible_for_Donation,0,1
row_0,,
0,3584,0
1,0,6416


Eligibility rule engine

In [11]:
def eligibility_rule_engine(row):

    if row["Medical_Condition"] != "None":
        return "Not Eligible: Medical condition restriction"

    if row["Weight_kg"] < 50:
        return "Not Eligible: Weight below threshold"

    if row["Hemoglobin_g_dL"] < 12.5:
        return "Not Eligible: Hemoglobin below threshold"

    return "Eligible"

Test it

In [12]:
df["Rule_Based_Eligibility"] = df.apply(
    eligibility_rule_engine,
    axis=1
)

df[[
    "Eligible_for_Donation",
    "Rule_Based_Eligibility"
]].head()

,Eligible_for_Donation,Rule_Based_Eligibility
0,1,Eligible
1,0,Not Eligible: Medical condition restriction
2,0,Not Eligible: Medical condition restriction
3,1,Eligible
4,1,Eligible


verify

In [13]:
(df["Rule_Based_Eligibility"] == "Eligible").astype(int).eq(
    df["Eligible_for_Donation"]
).mean()

np.float64(1.0)